# MLP para classificação de diabetes

Notebook autocontido. Toda a implementação está neste arquivo, organizada em funções pequenas, explícitas e executáveis em ordem.

## 1. Configuração

Valores do experimento e validações independentes.

In [ ]:
"""Configuração explícita do experimento."""

import os
from pathlib import Path


class ExperimentConfig:
    """Valores necessários para executar um experimento binário."""

    def __init__(
        self,
        project_root: Path,
        data_path: Path,
        artifacts_directory: Path,
        seed: int,
        training_fraction: float,
        validation_fraction: float,
        test_fraction: float,
        batch_size: int,
        epochs: int,
        minimum_epochs: int,
        f_beta: float,
        threshold_minimum: float,
        threshold_maximum: float,
        threshold_step: float,
        learning_rate: float,
        weight_decay: float,
        hidden_dimensions: list[int],
        dropout: float,
        output_size: int,
        early_stopping_patience: int,
        log_interval: int,
        requested_device: str | None,
        maximum_rows: int | None,
        use_class_weights: bool,
    ) -> None:
        self.project_root = project_root
        self.data_path = data_path
        self.artifacts_directory = artifacts_directory
        self.seed = seed
        self.training_fraction = training_fraction
        self.validation_fraction = validation_fraction
        self.test_fraction = test_fraction
        self.batch_size = batch_size
        self.epochs = epochs
        self.minimum_epochs = minimum_epochs
        self.f_beta = f_beta
        self.threshold_minimum = threshold_minimum
        self.threshold_maximum = threshold_maximum
        self.threshold_step = threshold_step
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.hidden_dimensions = hidden_dimensions
        self.dropout = dropout
        self.output_size = output_size
        self.early_stopping_patience = early_stopping_patience
        self.log_interval = log_interval
        self.requested_device = requested_device
        self.maximum_rows = maximum_rows
        self.use_class_weights = use_class_weights


def create_default_config(project_root: Path) -> ExperimentConfig:
    data_path = project_root / "dataset" / "diabetes_prediction_dataset.csv"
    artifacts_directory = project_root / "artifacts"
    hidden_dimensions = [32, 16]

    config = ExperimentConfig(
        project_root=project_root,
        data_path=data_path,
        artifacts_directory=artifacts_directory,
        seed=42,
        training_fraction=0.70,
        validation_fraction=0.15,
        test_fraction=0.15,
        batch_size=32,
        epochs=60,
        minimum_epochs=30,
        f_beta=2.0,
        threshold_minimum=0.40,
        threshold_maximum=0.60,
        threshold_step=0.01,
        learning_rate=1e-3,
        weight_decay=1e-4,
        hidden_dimensions=hidden_dimensions,
        dropout=0.30,
        output_size=1,
        early_stopping_patience=8,
        log_interval=1,
        requested_device=None,
        maximum_rows=None,
        use_class_weights=True,
    )
    return config


def read_optional_integer_environment_variable(name: str) -> int | None:
    value = os.getenv(name)
    if value is None:
        return None
    if value == "":
        return None
    return int(value)


def read_boolean_environment_variable(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    normalized_value = value.strip().lower()
    true_values = ["1", "true", "yes"]
    if normalized_value in true_values:
        return True
    return False


def apply_environment_overrides(config: ExperimentConfig) -> ExperimentConfig:
    artifacts_value = os.getenv("MLP_ARTIFACTS_DIR")
    if artifacts_value is not None:
        if artifacts_value != "":
            config.artifacts_directory = Path(artifacts_value)
    epochs_value = read_optional_integer_environment_variable("MLP_EPOCHS")
    if epochs_value is not None:
        config.epochs = epochs_value
    batch_size_value = read_optional_integer_environment_variable("MLP_BATCH_SIZE")
    if batch_size_value is not None:
        config.batch_size = batch_size_value
    maximum_rows_value = read_optional_integer_environment_variable("MLP_MAX_ROWS")
    config.maximum_rows = maximum_rows_value
    device_value = os.getenv("MLP_DEVICE")
    if device_value is not None:
        if device_value == "":
            config.requested_device = None
        else:
            config.requested_device = device_value
    class_weights_value = read_boolean_environment_variable(
        "MLP_CLASS_WEIGHTS",
        config.use_class_weights,
    )
    config.use_class_weights = class_weights_value
    return config


def validate_split_proportions(config: ExperimentConfig) -> None:
    total = config.training_fraction + config.validation_fraction + config.test_fraction
    if abs(total - 1.0) > 1e-9:
        raise ValueError("As frações de treino, validação e teste devem somar 1.0.")
    proportions = [config.training_fraction, config.validation_fraction, config.test_fraction]
    for proportion in proportions:
        if proportion <= 0:
            raise ValueError("Todas as frações devem ser positivas.")


def validate_training_values(config: ExperimentConfig) -> None:
    if config.batch_size <= 0:
        raise ValueError("O tamanho do lote deve ser positivo.")
    if config.epochs <= 0:
        raise ValueError("O número máximo de épocas deve ser positivo.")
    if config.minimum_epochs <= 0:
        raise ValueError("O número mínimo de épocas deve ser positivo.")
    if config.minimum_epochs > config.epochs:
        raise ValueError("O mínimo de épocas não pode superar o máximo.")
    if config.f_beta <= 0:
        raise ValueError("F-beta deve ser positivo.")
    if config.threshold_minimum <= 0:
        raise ValueError("O limiar mínimo deve ser maior que zero.")
    if config.threshold_maximum >= 1:
        raise ValueError("O limiar máximo deve ser menor que um.")
    if config.threshold_minimum >= config.threshold_maximum:
        raise ValueError("Os limites de limiar são inválidos.")
    if config.threshold_step <= 0:
        raise ValueError("O passo de limiar deve ser positivo.")
    if config.learning_rate <= 0:
        raise ValueError("A taxa de aprendizado deve ser positiva.")
    if config.weight_decay < 0:
        raise ValueError("O weight decay não pode ser negativo.")
    if config.early_stopping_patience <= 0:
        raise ValueError("A paciência do early stopping deve ser positiva.")
    if config.log_interval <= 0:
        raise ValueError("O intervalo de log deve ser positivo.")
    if config.maximum_rows is not None:
        if config.maximum_rows <= 0:
            raise ValueError("O limite de linhas deve ser positivo.")


def validate_model_values(config: ExperimentConfig) -> None:
    if config.output_size != 1:
        raise ValueError("A classificação binária exige exatamente uma saída.")
    if config.dropout < 0:
        raise ValueError("Dropout não pode ser negativo.")
    if config.dropout >= 1:
        raise ValueError("Dropout deve ser menor que 1.")
    if len(config.hidden_dimensions) == 0:
        raise ValueError("A MLP precisa de ao menos uma camada oculta.")
    for hidden_dimension in config.hidden_dimensions:
        if hidden_dimension <= 0:
            raise ValueError("As dimensões ocultas devem ser positivas.")


def validate_config(config: ExperimentConfig) -> None:
    validate_split_proportions(config)
    validate_training_values(config)
    validate_model_values(config)


def get_checkpoint_path(config: ExperimentConfig) -> Path:
    return config.artifacts_directory / "best_model.pt"


def get_preprocessor_path(config: ExperimentConfig) -> Path:
    return config.artifacts_directory / "preprocessor.joblib"


def config_to_dictionary(config: ExperimentConfig) -> dict[str, object]:
    values: dict[str, object] = {}
    values["project_root"] = str(config.project_root)
    values["data_path"] = str(config.data_path)
    values["artifacts_directory"] = str(config.artifacts_directory)
    values["seed"] = config.seed
    values["training_fraction"] = config.training_fraction
    values["validation_fraction"] = config.validation_fraction
    values["test_fraction"] = config.test_fraction
    values["batch_size"] = config.batch_size
    values["epochs"] = config.epochs
    values["minimum_epochs"] = config.minimum_epochs
    values["f_beta"] = config.f_beta
    values["threshold_minimum"] = config.threshold_minimum
    values["threshold_maximum"] = config.threshold_maximum
    values["threshold_step"] = config.threshold_step
    values["learning_rate"] = config.learning_rate
    values["weight_decay"] = config.weight_decay
    values["hidden_dimensions"] = list(config.hidden_dimensions)
    values["dropout"] = config.dropout
    values["output_size"] = config.output_size
    values["early_stopping_patience"] = config.early_stopping_patience
    values["log_interval"] = config.log_interval
    values["requested_device"] = config.requested_device
    values["maximum_rows"] = config.maximum_rows
    values["use_class_weights"] = config.use_class_weights
    return values


## 2. Ambiente e reprodutibilidade

Sementes, determinismo e seleção explícita de CPU ou CUDA.

In [ ]:
"""Seleção de dispositivo e reprodutibilidade."""

import platform
import random
import sys

import numpy as np
import torch


class RuntimeMetadata:
    def __init__(
        self,
        python_version: str,
        platform_name: str,
        torch_version: str,
        torch_cuda_version: str | None,
        device_name: str,
        cuda_available: bool,
        gpu_name: str | None,
        cuda_device_count: int | None,
    ) -> None:
        self.python_version = python_version
        self.platform_name = platform_name
        self.torch_version = torch_version
        self.torch_cuda_version = torch_cuda_version
        self.device_name = device_name
        self.cuda_available = cuda_available
        self.gpu_name = gpu_name
        self.cuda_device_count = cuda_device_count

    def to_dictionary(self) -> dict[str, object]:
        values: dict[str, object] = {}
        values["python"] = self.python_version
        values["platform"] = self.platform_name
        values["torch"] = self.torch_version
        values["torch_cuda_version"] = self.torch_cuda_version
        values["device"] = self.device_name
        values["cuda_available"] = self.cuda_available
        if self.gpu_name is not None:
            values["gpu_name"] = self.gpu_name
        if self.cuda_device_count is not None:
            values["cuda_device_count"] = self.cuda_device_count
        return values


def seed_python(seed: int) -> None:
    random.seed(seed)


def seed_numpy(seed: int) -> None:
    np.random.seed(seed)


def seed_torch(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def configure_deterministic_torch() -> None:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def configure_reproducibility(seed: int) -> None:
    seed_python(seed)
    seed_numpy(seed)
    seed_torch(seed)
    configure_deterministic_torch()


def validate_requested_device(device_name: str) -> None:
    requested_device = torch.device(device_name)
    if requested_device.type == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA foi solicitada, mas não está disponível.")


def select_device(device_name: str | None) -> torch.device:
    if device_name is not None:
        validate_requested_device(device_name)
        return torch.device(device_name)

    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def collect_runtime_metadata(device: torch.device) -> RuntimeMetadata:
    gpu_name = None
    cuda_device_count = None
    if device.type == "cuda":
        gpu_name = torch.cuda.get_device_name(device)
        cuda_device_count = torch.cuda.device_count()

    metadata = RuntimeMetadata(
        python_version=sys.version,
        platform_name=platform.platform(),
        torch_version=torch.__version__,
        torch_cuda_version=torch.version.cuda,
        device_name=str(device),
        cuda_available=torch.cuda.is_available(),
        gpu_name=gpu_name,
        cuda_device_count=cuda_device_count,
    )
    return metadata


def print_device_summary(metadata: RuntimeMetadata) -> None:
    print("Dispositivo selecionado: " + metadata.device_name)
    if metadata.gpu_name is not None:
        cuda_version = str(metadata.torch_cuda_version)
        print("GPU: " + metadata.gpu_name + " | CUDA PyTorch: " + cuda_version)
        return
    print("CUDA indisponível ou não solicitado; usando CPU.")



## 3. Leitura e divisão dos dados

Cada regra de validação do CSV possui uma função própria.

In [ ]:
"""Leitura, validação e divisão do dataset."""

from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split



CATEGORICAL_COLUMNS = ["gender", "smoking_history"]
NUMERIC_COLUMNS = [
    "age",
    "hypertension",
    "heart_disease",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level",
]
TARGET_COLUMN = "diabetes"
FEATURE_COLUMNS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS
REQUIRED_COLUMNS = FEATURE_COLUMNS + [TARGET_COLUMN]


class FeaturesAndTarget:
    def __init__(self, features: pd.DataFrame, target: pd.Series) -> None:
        self.features = features
        self.target = target


class RemainingAndTestData:
    def __init__(
        self,
        remaining_features: pd.DataFrame,
        test_features: pd.DataFrame,
        remaining_target: pd.Series,
        test_target: pd.Series,
    ) -> None:
        self.remaining_features = remaining_features
        self.test_features = test_features
        self.remaining_target = remaining_target
        self.test_target = test_target


class DatasetSplits:
    def __init__(
        self,
        x_train: pd.DataFrame,
        x_validation: pd.DataFrame,
        x_test: pd.DataFrame,
        y_train: pd.Series,
        y_validation: pd.Series,
        y_test: pd.Series,
    ) -> None:
        self.x_train = x_train
        self.x_validation = x_validation
        self.x_test = x_test
        self.y_train = y_train
        self.y_validation = y_validation
        self.y_test = y_test


class TargetSummary:
    def __init__(self, row_count: int, positive_rate: float, positive_count: int) -> None:
        self.row_count = row_count
        self.positive_rate = positive_rate
        self.positive_count = positive_count

    def to_dictionary(self) -> dict[str, float | int]:
        values: dict[str, float | int] = {}
        values["rows"] = self.row_count
        values["positive_rate"] = self.positive_rate
        values["positive_count"] = self.positive_count
        return values


class DatasetSplitSummary:
    def __init__(
        self,
        training: TargetSummary,
        validation: TargetSummary,
        test: TargetSummary,
    ) -> None:
        self.training = training
        self.validation = validation
        self.test = test

    def to_dictionary(self) -> dict[str, dict[str, float | int]]:
        values: dict[str, dict[str, float | int]] = {}
        values["train"] = self.training.to_dictionary()
        values["validation"] = self.validation.to_dictionary()
        values["test"] = self.test.to_dictionary()
        return values


class DatasetDiagnostics:
    def __init__(
        self,
        row_count: int,
        data_types: dict[str, str],
        null_counts: dict[str, int],
        target_distribution: dict[str, int],
    ) -> None:
        self.row_count = row_count
        self.data_types = data_types
        self.null_counts = null_counts
        self.target_distribution = target_distribution

    def to_dictionary(self) -> dict[str, object]:
        values: dict[str, object] = {}
        values["rows"] = self.row_count
        values["dtypes"] = self.data_types
        values["nulls"] = self.null_counts
        values["target_distribution"] = self.target_distribution
        return values


def ensure_dataset_file_exists(path: Path) -> None:
    if not path.is_file():
        raise FileNotFoundError("Dataset não encontrado: " + str(path))


def read_dataset_csv(path: Path, maximum_rows: int | None) -> pd.DataFrame:
    return pd.read_csv(path, nrows=maximum_rows)


def find_missing_columns(
    frame: pd.DataFrame,
    required_columns: list[str],
) -> list[str]:
    missing_columns: list[str] = []
    for required_column in required_columns:
        if required_column not in frame.columns:
            missing_columns.append(required_column)
    missing_columns.sort()
    return missing_columns


def ensure_required_columns_exist(frame: pd.DataFrame) -> None:
    missing_columns = find_missing_columns(frame, REQUIRED_COLUMNS)
    if len(missing_columns) == 0:
        return

    found_columns = list(frame.columns)
    message = "CSV incompatível; colunas ausentes: " + str(missing_columns)
    message = message + ". Encontradas: " + str(found_columns)
    raise ValueError(message)


def ensure_dataset_is_not_empty(frame: pd.DataFrame) -> None:
    if frame.empty:
        raise ValueError("O dataset está vazio.")


def find_null_counts(frame: pd.DataFrame) -> dict[str, int]:
    counts: dict[str, int] = {}
    null_counts = frame[REQUIRED_COLUMNS].isna().sum()
    for column in REQUIRED_COLUMNS:
        count = int(null_counts[column])
        if count > 0:
            counts[column] = count
    return counts


def ensure_dataset_has_no_nulls(frame: pd.DataFrame) -> None:
    null_counts = find_null_counts(frame)
    if len(null_counts) == 0:
        return

    message = "O dataset contém valores ausentes. "
    message = message + "Política configurada: rejeitar. Detalhes: "
    message = message + str(null_counts)
    raise ValueError(message)


def ensure_target_is_binary(frame: pd.DataFrame) -> None:
    target = frame[TARGET_COLUMN]
    unique_values = target.unique().tolist()
    invalid_values: list[object] = []
    for value in unique_values:
        if value != 0:
            if value != 1:
                invalid_values.append(value)
    if len(invalid_values) > 0:
        invalid_values.sort()
        raise ValueError(
            "O alvo deve conter somente 0 e 1; encontrados: " + str(invalid_values)
        )


def ensure_target_has_both_classes(frame: pd.DataFrame) -> None:
    target = frame[TARGET_COLUMN]
    if target.nunique() != 2:
        raise ValueError("A divisão estratificada requer as duas classes no alvo.")


def select_model_columns(frame: pd.DataFrame) -> pd.DataFrame:
    return frame[REQUIRED_COLUMNS].copy()


def load_validated_dataset(path: Path, maximum_rows: int | None) -> pd.DataFrame:
    ensure_dataset_file_exists(path)
    frame = read_dataset_csv(path, maximum_rows)
    ensure_required_columns_exist(frame)
    ensure_dataset_is_not_empty(frame)
    ensure_dataset_has_no_nulls(frame)
    ensure_target_is_binary(frame)
    ensure_target_has_both_classes(frame)
    return select_model_columns(frame)


def separate_features_and_target(frame: pd.DataFrame) -> FeaturesAndTarget:
    features = frame.drop(columns=TARGET_COLUMN)
    target = frame[TARGET_COLUMN].astype("int64")
    return FeaturesAndTarget(features, target)


def calculate_relative_validation_size(config: ExperimentConfig) -> float:
    remaining_fraction = config.training_fraction + config.validation_fraction
    return config.validation_fraction / remaining_fraction


def split_test_set(
    features_and_target: FeaturesAndTarget,
    config: ExperimentConfig,
) -> RemainingAndTestData:
    split_values = train_test_split(
        features_and_target.features,
        features_and_target.target,
        test_size=config.test_fraction,
        random_state=config.seed,
        stratify=features_and_target.target,
    )
    remaining_features = split_values[0]
    test_features = split_values[1]
    remaining_target = split_values[2]
    test_target = split_values[3]
    return RemainingAndTestData(
        remaining_features,
        test_features,
        remaining_target,
        test_target,
    )


def split_training_and_validation(
    remaining_and_test: RemainingAndTestData,
    config: ExperimentConfig,
) -> DatasetSplits:
    validation_size = calculate_relative_validation_size(config)
    split_values = train_test_split(
        remaining_and_test.remaining_features,
        remaining_and_test.remaining_target,
        test_size=validation_size,
        random_state=config.seed,
        stratify=remaining_and_test.remaining_target,
    )
    training_features = split_values[0]
    validation_features = split_values[1]
    training_target = split_values[2]
    validation_target = split_values[3]

    return DatasetSplits(
        x_train=training_features,
        x_validation=validation_features,
        x_test=remaining_and_test.test_features,
        y_train=training_target,
        y_validation=validation_target,
        y_test=remaining_and_test.test_target,
    )


def create_dataset_splits(
    frame: pd.DataFrame,
    config: ExperimentConfig,
) -> DatasetSplits:
    features_and_target = separate_features_and_target(frame)
    remaining_and_test = split_test_set(features_and_target, config)
    return split_training_and_validation(remaining_and_test, config)


def summarize_target(target: pd.Series) -> TargetSummary:
    row_count = int(len(target))
    positive_rate = float(target.mean())
    positive_count = int(target.sum())
    return TargetSummary(row_count, positive_rate, positive_count)


def summarize_dataset_splits(splits: DatasetSplits) -> DatasetSplitSummary:
    training_summary = summarize_target(splits.y_train)
    validation_summary = summarize_target(splits.y_validation)
    test_summary = summarize_target(splits.y_test)
    return DatasetSplitSummary(training_summary, validation_summary, test_summary)


def create_dataset_diagnostics(frame: pd.DataFrame) -> DatasetDiagnostics:
    data_types: dict[str, str] = {}
    null_counts: dict[str, int] = {}
    target_distribution: dict[str, int] = {}

    for column in frame.columns:
        data_types[column] = str(frame[column].dtype)
        null_counts[column] = int(frame[column].isna().sum())

    distribution = frame[TARGET_COLUMN].value_counts().sort_index()
    for label in distribution.index:
        target_distribution[str(label)] = int(distribution[label])

    return DatasetDiagnostics(
        row_count=int(len(frame)),
        data_types=data_types,
        null_counts=null_counts,
        target_distribution=target_distribution,
    )



## 4. Pré-processamento

O pré-processador é ajustado somente no treino; validação e teste apenas transformam.

In [ ]:
"""Pré-processamento e criação explícita dos DataLoaders binários."""

import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset


class TransformedSplits:
    def __init__(self, training: np.ndarray, validation: np.ndarray, test: np.ndarray) -> None:
        self.training = training
        self.validation = validation
        self.test = test


class DataLoaders:
    def __init__(self, train: DataLoader, validation: DataLoader, test: DataLoader) -> None:
        self.train = train
        self.validation = validation
        self.test = test


def create_categorical_transformer() -> OneHotEncoder:
    return OneHotEncoder(handle_unknown="ignore", sparse_output=False)


def create_numeric_transformer() -> Pipeline:
    steps = [("scaler", StandardScaler())]
    return Pipeline(steps)


def create_preprocessor() -> ColumnTransformer:
    categorical_transformer = create_categorical_transformer()
    numeric_transformer = create_numeric_transformer()
    transformers = [
        ("categorical", categorical_transformer, CATEGORICAL_COLUMNS),
        ("numeric", numeric_transformer, NUMERIC_COLUMNS),
    ]
    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)


def convert_matrix_to_float32(matrix: object) -> np.ndarray:
    return np.asarray(matrix, dtype=np.float32)


def fit_and_transform_training_data(preprocessor: ColumnTransformer, training_features: pd.DataFrame) -> np.ndarray:
    return convert_matrix_to_float32(preprocessor.fit_transform(training_features))


def transform_validation_data(preprocessor: ColumnTransformer, validation_features: pd.DataFrame) -> np.ndarray:
    return convert_matrix_to_float32(preprocessor.transform(validation_features))


def transform_test_data(preprocessor: ColumnTransformer, test_features: pd.DataFrame) -> np.ndarray:
    return convert_matrix_to_float32(preprocessor.transform(test_features))


def ensure_equal_feature_widths(transformed_splits: TransformedSplits) -> None:
    training_width = transformed_splits.training.shape[1]
    validation_width = transformed_splits.validation.shape[1]
    test_width = transformed_splits.test.shape[1]
    if training_width != validation_width or training_width != test_width:
        widths: dict[str, int] = {}
        widths["train"] = training_width
        widths["validation"] = validation_width
        widths["test"] = test_width
        raise RuntimeError("Dimensões transformadas incompatíveis: " + str(widths))


def transform_dataset_splits(splits: DatasetSplits, preprocessor: ColumnTransformer) -> TransformedSplits:
    training = fit_and_transform_training_data(preprocessor, splits.x_train)
    validation = transform_validation_data(preprocessor, splits.x_validation)
    test = transform_test_data(preprocessor, splits.x_test)
    transformed_splits = TransformedSplits(training, validation, test)
    ensure_equal_feature_widths(transformed_splits)
    return transformed_splits


def convert_labels_to_float32_column(target: pd.Series) -> np.ndarray:
    values = target.to_numpy(dtype=np.float32, copy=True)
    return values.reshape(-1, 1)


def create_tensor_dataset(features: np.ndarray, labels: np.ndarray) -> TensorDataset:
    feature_tensor = torch.from_numpy(features)
    label_tensor = torch.from_numpy(labels)
    return TensorDataset(feature_tensor, label_tensor)


def create_training_generator(seed: int) -> torch.Generator:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


def create_data_loader(dataset: TensorDataset, batch_size: int, shuffle: bool, pin_memory: bool, generator: torch.Generator | None) -> DataLoader:
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, pin_memory=pin_memory, generator=generator, num_workers=0)


def create_data_loaders(transformed_splits: TransformedSplits, splits: DatasetSplits, config: ExperimentConfig, device: torch.device) -> DataLoaders:
    training_labels = convert_labels_to_float32_column(splits.y_train)
    validation_labels = convert_labels_to_float32_column(splits.y_validation)
    test_labels = convert_labels_to_float32_column(splits.y_test)
    training_dataset = create_tensor_dataset(transformed_splits.training, training_labels)
    validation_dataset = create_tensor_dataset(transformed_splits.validation, validation_labels)
    test_dataset = create_tensor_dataset(transformed_splits.test, test_labels)
    pin_memory = device.type == "cuda"
    training_generator = create_training_generator(config.seed)
    training_loader = create_data_loader(training_dataset, config.batch_size, True, pin_memory, training_generator)
    validation_loader = create_data_loader(validation_dataset, config.batch_size, False, pin_memory, None)
    test_loader = create_data_loader(test_dataset, config.batch_size, False, pin_memory, None)
    return DataLoaders(training_loader, validation_loader, test_loader)


def get_transformed_feature_count(transformed_splits: TransformedSplits) -> int:
    return int(transformed_splits.training.shape[1])


## 5. Modelo

A MLP é montada em blocos Linear, BatchNorm, ReLU e Dropout.

In [ ]:
"""Definição explícita da MLP binária de saída única."""

import torch
from torch import nn


def validate_hidden_dimensions(hidden_dimensions: list[int]) -> None:
    if len(hidden_dimensions) == 0:
        raise ValueError("A MLP precisa de ao menos uma dimensão oculta.")
    for hidden_dimension in hidden_dimensions:
        if hidden_dimension <= 0:
            raise ValueError("As dimensões ocultas devem ser positivas.")


def create_hidden_block(input_size: int, output_size: int, dropout: float) -> list[nn.Module]:
    modules: list[nn.Module] = []
    modules.append(nn.Linear(input_size, output_size))
    modules.append(nn.BatchNorm1d(output_size))
    modules.append(nn.ReLU())
    modules.append(nn.Dropout(dropout))
    return modules


def create_network_layers(input_size: int, hidden_dimensions: list[int], output_size: int, dropout: float) -> list[nn.Module]:
    validate_hidden_dimensions(hidden_dimensions)
    if output_size != 1:
        raise ValueError("A MLP binária precisa ter uma única saída.")
    layers: list[nn.Module] = []
    previous_size = input_size
    for hidden_dimension in hidden_dimensions:
        hidden_block = create_hidden_block(previous_size, hidden_dimension, dropout)
        for module in hidden_block:
            layers.append(module)
        previous_size = hidden_dimension
    layers.append(nn.Linear(previous_size, output_size))
    return layers


class MLP(nn.Module):
    def __init__(self, input_size: int, hidden_dimensions: list[int], output_size: int, dropout: float) -> None:
        super().__init__()
        self.network = nn.Sequential(*create_network_layers(input_size, hidden_dimensions, output_size, dropout))
        self.input_size = input_size
        self.hidden_dimensions = list(hidden_dimensions)
        self.output_size = output_size
        self.dropout = dropout

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features)


def create_model(input_size: int, config: ExperimentConfig, device: torch.device) -> MLP:
    model = MLP(input_size, config.hidden_dimensions, config.output_size, config.dropout)
    model.to(device)
    return model


def validate_model_output(model: MLP, sample_features: torch.Tensor, output_size: int, device: torch.device) -> None:
    model.eval()
    features_on_device = sample_features.to(device)
    with torch.no_grad():
        logits = model(features_on_device)
    expected_shape = (sample_features.shape[0], output_size)
    if logits.shape != expected_shape:
        message = "Formato de saída inválido. Esperado: " + str(expected_shape)
        message = message + ". Recebido: " + str(tuple(logits.shape))
        raise RuntimeError(message)
    if not torch.isfinite(logits).all():
        raise RuntimeError("A saída do modelo contém valores não finitos.")


## 6. Treinamento

Lote, época, checkpoint e early stopping são responsabilidades separadas.

In [ ]:
"""Treino binário, F2, calibração de limiar e checkpoint."""

from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, fbeta_score, precision_score, recall_score
from torch import nn
from torch.utils.data import DataLoader


class DeviceBatch:
    def __init__(self, features: torch.Tensor, labels: torch.Tensor) -> None:
        self.features = features
        self.labels = labels


class BatchTrainingResult:
    def __init__(self, weighted_loss: float, example_count: int) -> None:
        self.weighted_loss = weighted_loss
        self.example_count = example_count


class BatchPrediction:
    def __init__(self, weighted_loss: float, example_count: int, actual_labels: list[int], positive_probabilities: list[float]) -> None:
        self.weighted_loss = weighted_loss
        self.example_count = example_count
        self.actual_labels = actual_labels
        self.positive_probabilities = positive_probabilities


class EpochResult:
    def __init__(self, average_loss: float, actual_labels: np.ndarray | None, positive_probabilities: np.ndarray | None) -> None:
        self.average_loss = average_loss
        self.actual_labels = actual_labels
        self.positive_probabilities = positive_probabilities


class PositiveClassMetrics:
    def __init__(self, accuracy: float, precision: float, recall: float, f1: float, f2: float) -> None:
        self.accuracy = accuracy
        self.precision = precision
        self.recall = recall
        self.f1 = f1
        self.f2 = f2

    def to_dictionary(self) -> dict[str, float]:
        values: dict[str, float] = {}
        values["accuracy"] = self.accuracy
        values["precision_positive"] = self.precision
        values["recall_positive"] = self.recall
        values["f1_positive"] = self.f1
        values["f2_positive"] = self.f2
        return values


class ThresholdSelection:
    def __init__(self, threshold: float, metrics: PositiveClassMetrics) -> None:
        self.threshold = threshold
        self.metrics = metrics

    def to_dictionary(self) -> dict[str, float]:
        values = self.metrics.to_dictionary()
        values["positive_threshold"] = self.threshold
        return values


class TrainingHistory:
    def __init__(self) -> None:
        self.training_losses: list[float] = []
        self.validation_losses: list[float] = []
        self.validation_accuracies: list[float] = []
        self.validation_precisions: list[float] = []
        self.validation_recalls: list[float] = []
        self.validation_f1_scores: list[float] = []
        self.validation_f2_scores: list[float] = []
        self.validation_thresholds: list[float] = []

    def add_epoch(self, training_loss: float, validation_loss: float, selection: ThresholdSelection) -> None:
        self.training_losses.append(training_loss)
        self.validation_losses.append(validation_loss)
        self.validation_accuracies.append(selection.metrics.accuracy)
        self.validation_precisions.append(selection.metrics.precision)
        self.validation_recalls.append(selection.metrics.recall)
        self.validation_f1_scores.append(selection.metrics.f1)
        self.validation_f2_scores.append(selection.metrics.f2)
        self.validation_thresholds.append(selection.threshold)

    def to_dictionary(self) -> dict[str, list[float]]:
        values: dict[str, list[float]] = {}
        values["train_loss"] = list(self.training_losses)
        values["val_loss"] = list(self.validation_losses)
        values["val_acc"] = list(self.validation_accuracies)
        values["val_precision"] = list(self.validation_precisions)
        values["val_recall"] = list(self.validation_recalls)
        values["val_f1"] = list(self.validation_f1_scores)
        values["val_f2"] = list(self.validation_f2_scores)
        values["val_threshold"] = list(self.validation_thresholds)
        return values


class TrainingOutcome:
    def __init__(self, history: TrainingHistory, validation_selection: ThresholdSelection) -> None:
        self.history = history
        self.validation_selection = validation_selection


def move_batch_to_device(batch: list[torch.Tensor] | tuple[torch.Tensor, torch.Tensor], device: torch.device) -> DeviceBatch:
    non_blocking = device.type == "cuda"
    features = batch[0].to(device, non_blocking=non_blocking)
    labels = batch[1].to(device, non_blocking=non_blocking)
    return DeviceBatch(features, labels)


def calculate_batch_size(labels: torch.Tensor) -> int:
    return int(labels.size(0))


def train_single_batch(model: nn.Module, batch: list[torch.Tensor] | tuple[torch.Tensor, torch.Tensor], loss_function: nn.Module, optimizer: torch.optim.Optimizer, device: torch.device) -> BatchTrainingResult:
    device_batch = move_batch_to_device(batch, device)
    optimizer.zero_grad(set_to_none=True)
    logits = model(device_batch.features)
    loss = loss_function(logits, device_batch.labels)
    loss.backward()
    optimizer.step()
    example_count = calculate_batch_size(device_batch.labels)
    return BatchTrainingResult(float(loss.item()) * example_count, example_count)


def train_one_epoch(model: nn.Module, loader: DataLoader, loss_function: nn.Module, optimizer: torch.optim.Optimizer, device: torch.device) -> EpochResult:
    model.train()
    total_weighted_loss = 0.0
    total_examples = 0
    for batch in loader:
        result = train_single_batch(model, batch, loss_function, optimizer, device)
        total_weighted_loss = total_weighted_loss + result.weighted_loss
        total_examples = total_examples + result.example_count
    if total_examples == 0:
        raise RuntimeError("Não é possível treinar com um DataLoader vazio.")
    return EpochResult(total_weighted_loss / total_examples, None, None)


def convert_logits_to_positive_probabilities(logits: torch.Tensor) -> torch.Tensor:
    probabilities = torch.sigmoid(logits)
    return probabilities.reshape(-1)


def predict_single_batch(model: nn.Module, batch: list[torch.Tensor] | tuple[torch.Tensor, torch.Tensor], loss_function: nn.Module, device: torch.device) -> BatchPrediction:
    device_batch = move_batch_to_device(batch, device)
    logits = model(device_batch.features)
    loss = loss_function(logits, device_batch.labels)
    positive_probabilities = convert_logits_to_positive_probabilities(logits)
    example_count = calculate_batch_size(device_batch.labels)
    actual_labels = device_batch.labels.reshape(-1).to(torch.int64).cpu().tolist()
    probability_values = positive_probabilities.cpu().tolist()
    return BatchPrediction(float(loss.item()) * example_count, example_count, actual_labels, probability_values)


def evaluate_one_epoch(model: nn.Module, loader: DataLoader, loss_function: nn.Module, device: torch.device) -> EpochResult:
    model.eval()
    total_weighted_loss = 0.0
    total_examples = 0
    actual_labels: list[int] = []
    positive_probabilities: list[float] = []
    with torch.no_grad():
        for batch in loader:
            result = predict_single_batch(model, batch, loss_function, device)
            total_weighted_loss = total_weighted_loss + result.weighted_loss
            total_examples = total_examples + result.example_count
            actual_labels.extend(result.actual_labels)
            positive_probabilities.extend(result.positive_probabilities)
    if total_examples == 0:
        raise RuntimeError("Não é possível avaliar com um DataLoader vazio.")
    actual_array = np.asarray(actual_labels, dtype=np.int64)
    probability_array = np.asarray(positive_probabilities, dtype=np.float32)
    return EpochResult(total_weighted_loss / total_examples, actual_array, probability_array)


def create_predictions_from_threshold(positive_probabilities: np.ndarray, threshold: float) -> np.ndarray:
    predictions = np.zeros(len(positive_probabilities), dtype=np.int64)
    for index in range(len(positive_probabilities)):
        if float(positive_probabilities[index]) >= threshold:
            predictions[index] = 1
    return predictions


def calculate_positive_class_metrics(actual_labels: np.ndarray, predicted_labels: np.ndarray, f_beta: float) -> PositiveClassMetrics:
    accuracy = float(accuracy_score(actual_labels, predicted_labels))
    precision = float(precision_score(actual_labels, predicted_labels, pos_label=1, zero_division=0))
    recall = float(recall_score(actual_labels, predicted_labels, pos_label=1, zero_division=0))
    f1 = float(f1_score(actual_labels, predicted_labels, pos_label=1, zero_division=0))
    f2 = float(fbeta_score(actual_labels, predicted_labels, beta=f_beta, pos_label=1, zero_division=0))
    return PositiveClassMetrics(accuracy, precision, recall, f1, f2)


def create_threshold_values(config: ExperimentConfig) -> list[float]:
    values: list[float] = []
    threshold = config.threshold_minimum
    limit = config.threshold_maximum + (config.threshold_step / 1000)
    while threshold <= limit:
        values.append(round(threshold, 10))
        threshold = threshold + config.threshold_step
    return values


def is_better_threshold(candidate: ThresholdSelection, current_best: ThresholdSelection | None) -> bool:
    if current_best is None:
        return True
    if candidate.metrics.f2 > current_best.metrics.f2 + 1e-12:
        return True
    if abs(candidate.metrics.f2 - current_best.metrics.f2) > 1e-12:
        return False
    candidate_distance = abs(candidate.threshold - 0.50)
    current_distance = abs(current_best.threshold - 0.50)
    return candidate_distance < current_distance


def select_best_threshold(epoch_result: EpochResult, config: ExperimentConfig) -> ThresholdSelection:
    if epoch_result.actual_labels is None or epoch_result.positive_probabilities is None:
        raise ValueError("A seleção de limiar exige rótulos e probabilidades.")
    best_selection: ThresholdSelection | None = None
    for threshold in create_threshold_values(config):
        predicted_labels = create_predictions_from_threshold(epoch_result.positive_probabilities, threshold)
        metrics = calculate_positive_class_metrics(epoch_result.actual_labels, predicted_labels, config.f_beta)
        candidate = ThresholdSelection(threshold, metrics)
        if is_better_threshold(candidate, best_selection):
            best_selection = candidate
    if best_selection is None:
        raise RuntimeError("Nenhum limiar foi testado.")
    return best_selection


def calculate_positive_class_weight(training_target: pd.Series) -> torch.Tensor:
    target_values = training_target.to_numpy(dtype=np.int64, copy=True)
    positive_count = int(np.sum(target_values == 1))
    negative_count = int(np.sum(target_values == 0))
    if positive_count == 0 or negative_count == 0:
        raise ValueError("O treino precisa conter exemplos das duas classes.")
    positive_weight = negative_count / positive_count
    return torch.tensor([positive_weight], dtype=torch.float32)


def create_loss_function(positive_class_weight: torch.Tensor | None, device: torch.device) -> nn.BCEWithLogitsLoss:
    weight_on_device = None
    if positive_class_weight is not None:
        weight_on_device = positive_class_weight.to(device)
    return nn.BCEWithLogitsLoss(pos_weight=weight_on_device)


def create_optimizer(model: nn.Module, config: ExperimentConfig) -> torch.optim.Optimizer:
    return torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)


def validation_f2_improved(current_f2: float, best_f2: float) -> bool:
    return current_f2 > best_f2 + 1e-12


def should_stop_early(epoch: int, epochs_without_improvement: int, minimum_epochs: int, patience: int) -> bool:
    if epoch < minimum_epochs:
        return False
    return epochs_without_improvement >= patience


def create_checkpoint_data(model: MLP, epoch: int, selection: ThresholdSelection, history: TrainingHistory, config: ExperimentConfig) -> dict[str, object]:
    checkpoint: dict[str, object] = {}
    checkpoint["state_dict"] = model.state_dict()
    checkpoint["in_dim"] = model.input_size
    checkpoint["hidden_dims"] = list(model.hidden_dimensions)
    checkpoint["output_size"] = model.output_size
    checkpoint["dropout"] = model.dropout
    checkpoint["epoch"] = epoch
    checkpoint["positive_threshold"] = selection.threshold
    checkpoint["validation_selection"] = selection.to_dictionary()
    checkpoint["history"] = history.to_dictionary()
    checkpoint["config"] = config_to_dictionary(config)
    return checkpoint


def save_best_checkpoint(checkpoint: dict[str, object], checkpoint_path: Path) -> None:
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(checkpoint, checkpoint_path)


def create_selection_from_dictionary(values: dict[str, object]) -> ThresholdSelection:
    metrics = PositiveClassMetrics(
        float(values["accuracy"]), float(values["precision_positive"]), float(values["recall_positive"]),
        float(values["f1_positive"]), float(values["f2_positive"]),
    )
    return ThresholdSelection(float(values["positive_threshold"]), metrics)


def restore_best_model(model: MLP, checkpoint_path: Path, device: torch.device) -> dict[str, object]:
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    output_size = int(checkpoint["output_size"])
    if output_size != 1:
        raise ValueError("O checkpoint não usa o contrato de saída única.")
    model.load_state_dict(checkpoint["state_dict"])
    model.to(device)
    return checkpoint


def print_epoch_summary(epoch: int, training_loss: float, validation_loss: float, selection: ThresholdSelection) -> None:
    print("epoch=" + format(epoch, "03d") + " train_loss=" + format(training_loss, ".5f") + " val_loss=" + format(validation_loss, ".5f") + " threshold=" + format(selection.threshold, ".2f") + " val_recall=" + format(selection.metrics.recall, ".4f") + " val_f1=" + format(selection.metrics.f1, ".4f") + " val_f2=" + format(selection.metrics.f2, ".4f"))


def print_early_stopping(epoch: int, patience: int) -> None:
    print("Early stopping na época " + str(epoch) + "; paciência=" + str(patience) + ".")


def train_model(model: MLP, training_loader: DataLoader, validation_loader: DataLoader, loss_function: nn.Module, optimizer: torch.optim.Optimizer, config: ExperimentConfig, device: torch.device, checkpoint_path: Path) -> TrainingOutcome:
    history = TrainingHistory()
    best_validation_f2 = float("-inf")
    epochs_without_improvement = 0
    for epoch in range(1, config.epochs + 1):
        training_result = train_one_epoch(model, training_loader, loss_function, optimizer, device)
        validation_result = evaluate_one_epoch(model, validation_loader, loss_function, device)
        selection = select_best_threshold(validation_result, config)
        history.add_epoch(training_result.average_loss, validation_result.average_loss, selection)
        if epoch % config.log_interval == 0:
            print_epoch_summary(epoch, training_result.average_loss, validation_result.average_loss, selection)
        if validation_f2_improved(selection.metrics.f2, best_validation_f2):
            best_validation_f2 = selection.metrics.f2
            epochs_without_improvement = 0
            checkpoint = create_checkpoint_data(model, epoch, selection, history, config)
            save_best_checkpoint(checkpoint, checkpoint_path)
        else:
            epochs_without_improvement = epochs_without_improvement + 1
        if should_stop_early(epoch, epochs_without_improvement, config.minimum_epochs, config.early_stopping_patience):
            print_early_stopping(epoch, config.early_stopping_patience)
            break
    checkpoint = restore_best_model(model, checkpoint_path, device)
    selection_values = checkpoint["validation_selection"]
    selection = create_selection_from_dictionary(selection_values)
    return TrainingOutcome(history, selection)


## 7. Avaliação

As métricas são calculadas a partir de um único percurso pelo DataLoader.

In [ ]:
"""Cálculo das métricas finais no limiar calibrado."""

import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch import nn
from torch.utils.data import DataLoader


class ClassificationMetrics:
    def __init__(
        self,
        loss: float,
        positive_threshold: float,
        positive_metrics: PositiveClassMetrics,
        report_dictionary: dict[str, object],
        report_text: str,
        confusion: np.ndarray,
    ) -> None:
        self.loss = loss
        self.positive_threshold = positive_threshold
        self.positive_metrics = positive_metrics
        self.report_dictionary = report_dictionary
        self.report_text = report_text
        self.confusion = confusion

    def to_dictionary(self) -> dict[str, object]:
        values: dict[str, object] = {}
        values["loss"] = self.loss
        values["positive_threshold"] = self.positive_threshold
        metric_values = self.positive_metrics.to_dictionary()
        for key in metric_values:
            values[key] = metric_values[key]
        values["classification_report"] = self.report_dictionary
        return values


def create_classification_report_dictionary(
    actual: np.ndarray,
    predicted: np.ndarray,
) -> dict[str, object]:
    return classification_report(actual, predicted, output_dict=True, zero_division=0)


def create_classification_report_text(actual: np.ndarray, predicted: np.ndarray) -> str:
    return classification_report(actual, predicted, zero_division=0)


def create_confusion_matrix(actual: np.ndarray, predicted: np.ndarray) -> np.ndarray:
    return confusion_matrix(actual, predicted, labels=[0, 1])


def calculate_classification_metrics(
    epoch_result: EpochResult,
    positive_threshold: float,
    f_beta: float,
) -> ClassificationMetrics:
    if epoch_result.actual_labels is None:
        raise ValueError("O resultado não contém rótulos reais.")
    if epoch_result.positive_probabilities is None:
        raise ValueError("O resultado não contém probabilidades positivas.")
    predicted = create_predictions_from_threshold(
        epoch_result.positive_probabilities,
        positive_threshold,
    )
    positive_metrics = calculate_positive_class_metrics(
        epoch_result.actual_labels,
        predicted,
        f_beta,
    )
    report_dictionary = create_classification_report_dictionary(
        epoch_result.actual_labels,
        predicted,
    )
    report_text = create_classification_report_text(epoch_result.actual_labels, predicted)
    confusion = create_confusion_matrix(epoch_result.actual_labels, predicted)
    return ClassificationMetrics(
        epoch_result.average_loss,
        positive_threshold,
        positive_metrics,
        report_dictionary,
        report_text,
        confusion,
    )


def evaluate_test_set(
    model: nn.Module,
    loader: DataLoader,
    loss_function: nn.Module,
    device: torch.device,
    positive_threshold: float,
    f_beta: float,
) -> ClassificationMetrics:
    epoch_result = evaluate_one_epoch(model, loader, loss_function, device)
    return calculate_classification_metrics(epoch_result, positive_threshold, f_beta)


## 8. Artefatos

Cada arquivo ou gráfico possui uma operação de persistência nomeada.

In [ ]:
"""Persistência dos artefatos do experimento calibrado."""

import json
from pathlib import Path

import joblib
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer


def ensure_artifacts_directory_exists(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def save_json_file(path: Path, value: dict[str, object]) -> None:
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")


def save_text_file(path: Path, text: str) -> None:
    path.write_text(text, encoding="utf-8")


def save_preprocessor(path: Path, preprocessor: ColumnTransformer) -> None:
    joblib.dump(preprocessor, path)


def load_preprocessor(path: Path) -> ColumnTransformer:
    return joblib.load(path)


def save_test_metrics(metrics: ClassificationMetrics, artifacts_directory: Path) -> None:
    save_json_file(artifacts_directory / "test_metrics.json", metrics.to_dictionary())


def save_threshold_selection(
    selection: ThresholdSelection,
    artifacts_directory: Path,
) -> None:
    save_json_file(artifacts_directory / "threshold_selection.json", selection.to_dictionary())


def save_classification_report(metrics: ClassificationMetrics, artifacts_directory: Path) -> None:
    save_text_file(artifacts_directory / "classification_report.txt", metrics.report_text)


def save_confusion_matrix_figure(metrics: ClassificationMetrics, artifacts_directory: Path) -> None:
    figure, axis = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        metrics.confusion,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[0, 1],
        yticklabels=[0, 1],
        ax=axis,
    )
    axis.set_xlabel("Predito")
    axis.set_ylabel("Real")
    axis.set_title("Matriz de confusão — teste")
    figure.tight_layout()
    figure.savefig(artifacts_directory / "confusion_matrix.png", dpi=160)
    plt.close(figure)


def save_learning_curves_figure(history: TrainingHistory, artifacts_directory: Path) -> None:
    epochs = range(1, len(history.training_losses) + 1)
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    loss_axis = axes[0]
    metrics_axis = axes[1]
    loss_axis.plot(epochs, history.training_losses, label="Treino")
    loss_axis.plot(epochs, history.validation_losses, label="Validação")
    loss_axis.set_title("Perda")
    loss_axis.set_xlabel("Época")
    loss_axis.set_ylabel("Cross-entropy")
    metrics_axis.plot(epochs, history.validation_recalls, label="Recall positivo")
    metrics_axis.plot(epochs, history.validation_f1_scores, label="F1 positiva")
    metrics_axis.plot(epochs, history.validation_f2_scores, label="F2 positiva")
    metrics_axis.set_title("Métricas da classe positiva")
    metrics_axis.set_xlabel("Época")
    metrics_axis.set_ylabel("Valor")
    for axis in axes:
        axis.grid(True, alpha=0.3)
        axis.legend()
    figure.tight_layout()
    figure.savefig(artifacts_directory / "learning_curves.png", dpi=160)
    plt.close(figure)


def save_history(history: TrainingHistory, artifacts_directory: Path) -> None:
    values: dict[str, object] = {}
    history_values = history.to_dictionary()
    for key in history_values:
        values[key] = history_values[key]
    save_json_file(artifacts_directory / "history.json", values)


def save_metadata(metadata: dict[str, object], artifacts_directory: Path) -> None:
    save_json_file(artifacts_directory / "metadata.json", metadata)


def build_imbalance_report(metrics: ClassificationMetrics) -> str:
    lines: list[str] = []
    lines.append("Limiar positivo: " + format(metrics.positive_threshold, ".2f"))
    lines.append("Acurácia: " + format(metrics.positive_metrics.accuracy, ".5f"))
    lines.append("Precision positiva: " + format(metrics.positive_metrics.precision, ".5f"))
    lines.append("Recall positivo: " + format(metrics.positive_metrics.recall, ".5f"))
    lines.append("F1 positiva: " + format(metrics.positive_metrics.f1, ".5f"))
    lines.append("F2 positiva: " + format(metrics.positive_metrics.f2, ".5f"))
    lines.append("Conclusão: recall e F2 orientam a detecção da classe minoritária.")
    return "\n".join(lines) + "\n"


def save_imbalance_report(metrics: ClassificationMetrics, artifacts_directory: Path) -> None:
    save_text_file(artifacts_directory / "imbalance_report.txt", build_imbalance_report(metrics))


def save_experiment_artifacts(
    preprocessor: ColumnTransformer,
    preprocessor_path: Path,
    metrics: ClassificationMetrics,
    history: TrainingHistory,
    selection: ThresholdSelection,
    artifacts_directory: Path,
) -> None:
    ensure_artifacts_directory_exists(artifacts_directory)
    save_preprocessor(preprocessor_path, preprocessor)
    save_test_metrics(metrics, artifacts_directory)
    save_threshold_selection(selection, artifacts_directory)
    save_classification_report(metrics, artifacts_directory)
    save_confusion_matrix_figure(metrics, artifacts_directory)
    save_learning_curves_figure(history, artifacts_directory)
    save_history(history, artifacts_directory)
    save_imbalance_report(metrics, artifacts_directory)


## 9. Inferência

Modelo e pré-processador são carregados uma vez e reutilizados.

In [ ]:
"""Inferência binária com sigmoid e limiar salvo."""

from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer


class LoadedModel:
    def __init__(self, model: MLP, positive_threshold: float) -> None:
        self.model = model
        self.positive_threshold = positive_threshold


class Predictor:
    def __init__(self, model: MLP, preprocessor: ColumnTransformer, device: torch.device, positive_threshold: float) -> None:
        self.model = model
        self.preprocessor = preprocessor
        self.device = device
        self.positive_threshold = positive_threshold


def ensure_inference_columns_exist(frame: pd.DataFrame) -> None:
    missing_columns = find_missing_columns(frame, FEATURE_COLUMNS)
    if len(missing_columns) > 0:
        raise ValueError("Entradas sem colunas exigidas: " + str(missing_columns))


def select_inference_columns(frame: pd.DataFrame) -> pd.DataFrame:
    return frame[FEATURE_COLUMNS]


def load_model_from_checkpoint(path: Path, device: torch.device) -> LoadedModel:
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    if "output_size" not in checkpoint:
        raise ValueError("Checkpoint antigo de duas saídas não é compatível.")
    output_size = int(checkpoint["output_size"])
    if output_size != 1:
        raise ValueError("O checkpoint não usa uma saída binária única.")
    model = MLP(int(checkpoint["in_dim"]), list(checkpoint["hidden_dims"]), output_size, float(checkpoint["dropout"]))
    model.to(device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    threshold = float(checkpoint["positive_threshold"])
    return LoadedModel(model, threshold)


def transform_inference_features(frame: pd.DataFrame, preprocessor: ColumnTransformer) -> np.ndarray:
    transformed = preprocessor.transform(select_inference_columns(frame))
    return np.asarray(transformed, dtype=np.float32)


def predict_classes(model: MLP, transformed_features: np.ndarray, device: torch.device, positive_threshold: float) -> np.ndarray:
    features = torch.from_numpy(transformed_features).to(device)
    with torch.no_grad():
        logits = model(features)
        probabilities = convert_logits_to_positive_probabilities(logits)
    probability_values = probabilities.cpu().numpy()
    return create_predictions_from_threshold(probability_values, positive_threshold)


def load_predictor(checkpoint_path: Path, preprocessor_path: Path, device: torch.device) -> Predictor:
    loaded_model = load_model_from_checkpoint(checkpoint_path, device)
    preprocessor = load_preprocessor(preprocessor_path)
    return Predictor(loaded_model.model, preprocessor, device, loaded_model.positive_threshold)


def predict(predictor: Predictor, frame: pd.DataFrame) -> np.ndarray:
    ensure_inference_columns_exist(frame)
    features = transform_inference_features(frame, predictor.preprocessor)
    return predict_classes(predictor.model, features, predictor.device, predictor.positive_threshold)


## 10. Orquestração

O fluxo principal apenas coordena as funções definidas anteriormente.

In [ ]:
"""Orquestração do experimento binário calibrado por F2."""

import json
from pathlib import Path

import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from torch import nn
from torch.utils.data import DataLoader


class PreparedExperiment:
    def __init__(self, config: ExperimentConfig, device: torch.device, runtime_metadata: RuntimeMetadata) -> None:
        self.config = config
        self.device = device
        self.runtime_metadata = runtime_metadata


class PreparedTrainingData:
    def __init__(self, frame: pd.DataFrame, splits: DatasetSplits, split_summary: DatasetSplitSummary, diagnostics: DatasetDiagnostics, preprocessor: ColumnTransformer, loaders: DataLoaders, input_size: int) -> None:
        self.frame = frame
        self.splits = splits
        self.split_summary = split_summary
        self.diagnostics = diagnostics
        self.preprocessor = preprocessor
        self.loaders = loaders
        self.input_size = input_size


class TrainingComponents:
    def __init__(self, model: MLP, loss_function: nn.Module, optimizer: torch.optim.Optimizer, positive_class_weight: float | None) -> None:
        self.model = model
        self.loss_function = loss_function
        self.optimizer = optimizer
        self.positive_class_weight = positive_class_weight


class ExperimentResult:
    def __init__(self, config: ExperimentConfig, runtime_metadata: RuntimeMetadata, diagnostics: DatasetDiagnostics, split_summary: DatasetSplitSummary, training_outcome: TrainingOutcome, test_metrics: ClassificationMetrics, inference_predictions: list[int], input_size: int, positive_class_weight: float | None) -> None:
        self.config = config
        self.runtime_metadata = runtime_metadata
        self.diagnostics = diagnostics
        self.split_summary = split_summary
        self.training_outcome = training_outcome
        self.test_metrics = test_metrics
        self.inference_predictions = inference_predictions
        self.input_size = input_size
        self.positive_class_weight = positive_class_weight

    def to_dictionary(self) -> dict[str, object]:
        values = self.runtime_metadata.to_dictionary()
        values["config"] = config_to_dictionary(self.config)
        values["diagnostics"] = self.diagnostics.to_dictionary()
        values["splits"] = self.split_summary.to_dictionary()
        values["history"] = self.training_outcome.history.to_dictionary()
        values["validation_threshold_selection"] = self.training_outcome.validation_selection.to_dictionary()
        values["test_metrics"] = self.test_metrics.to_dictionary()
        values["inference_smoke_predictions"] = list(self.inference_predictions)
        values["class_weighting_enabled"] = self.config.use_class_weights
        values["positive_class_weight"] = self.positive_class_weight
        values["in_dim"] = self.input_size
        values["selection_reason"] = "Maior F2 de validação com limiar calibrado."
        return values


def prepare_experiment(config: ExperimentConfig) -> PreparedExperiment:
    validate_config(config)
    configure_reproducibility(config.seed)
    device = select_device(config.requested_device)
    metadata = collect_runtime_metadata(device)
    print_device_summary(metadata)
    ensure_artifacts_directory_exists(config.artifacts_directory)
    return PreparedExperiment(config, device, metadata)


def prepare_training_data(config: ExperimentConfig, device: torch.device) -> PreparedTrainingData:
    frame = load_validated_dataset(config.data_path, config.maximum_rows)
    diagnostics = create_dataset_diagnostics(frame)
    splits = create_dataset_splits(frame, config)
    split_summary = summarize_dataset_splits(splits)
    preprocessor = create_preprocessor()
    transformed_splits = transform_dataset_splits(splits, preprocessor)
    input_size = get_transformed_feature_count(transformed_splits)
    loaders = create_data_loaders(transformed_splits, splits, config, device)
    return PreparedTrainingData(frame, splits, split_summary, diagnostics, preprocessor, loaders, input_size)


def print_data_summary(training_data: PreparedTrainingData) -> None:
    print("Partições: " + str(training_data.split_summary.to_dictionary()) + "; in_dim=" + str(training_data.input_size))


def get_sample_features(training_loader: DataLoader) -> torch.Tensor:
    return next(iter(training_loader))[0]


def validate_prepared_model(model: MLP, training_data: PreparedTrainingData, config: ExperimentConfig, device: torch.device) -> None:
    validate_model_output(model, get_sample_features(training_data.loaders.train), config.output_size, device)
    if device.type == "cuda" and not training_data.loaders.train.pin_memory:
        raise RuntimeError("pin_memory deveria estar ativo em CUDA.")


def convert_weight_tensor_to_float(weight: torch.Tensor | None) -> float | None:
    if weight is None:
        return None
    return float(weight.item())


def prepare_training_components(training_data: PreparedTrainingData, config: ExperimentConfig, device: torch.device) -> TrainingComponents:
    model = create_model(training_data.input_size, config, device)
    validate_prepared_model(model, training_data, config, device)
    positive_class_weight = None
    if config.use_class_weights:
        positive_class_weight = calculate_positive_class_weight(training_data.splits.y_train)
    loss_function = create_loss_function(positive_class_weight, device)
    optimizer = create_optimizer(model, config)
    return TrainingComponents(model, loss_function, optimizer, convert_weight_tensor_to_float(positive_class_weight))


def run_training(components: TrainingComponents, training_data: PreparedTrainingData, prepared_experiment: PreparedExperiment) -> TrainingOutcome:
    return train_model(components.model, training_data.loaders.train, training_data.loaders.validation, components.loss_function, components.optimizer, prepared_experiment.config, prepared_experiment.device, get_checkpoint_path(prepared_experiment.config))


def run_test_evaluation(components: TrainingComponents, training_data: PreparedTrainingData, outcome: TrainingOutcome, config: ExperimentConfig, device: torch.device) -> ClassificationMetrics:
    return evaluate_test_set(components.model, training_data.loaders.test, components.loss_function, device, outcome.validation_selection.threshold, config.f_beta)


def persist_primary_artifacts(training_data: PreparedTrainingData, outcome: TrainingOutcome, metrics: ClassificationMetrics, config: ExperimentConfig) -> None:
    save_experiment_artifacts(training_data.preprocessor, get_preprocessor_path(config), metrics, outcome.history, outcome.validation_selection, config.artifacts_directory)


def verify_saved_inference(training_data: PreparedTrainingData, config: ExperimentConfig, device: torch.device) -> list[int]:
    predictor = load_predictor(get_checkpoint_path(config), get_preprocessor_path(config), device)
    return predict(predictor, training_data.splits.x_validation.head(3)).tolist()


def build_experiment_result(prepared_experiment: PreparedExperiment, training_data: PreparedTrainingData, components: TrainingComponents, outcome: TrainingOutcome, metrics: ClassificationMetrics, predictions: list[int]) -> ExperimentResult:
    return ExperimentResult(prepared_experiment.config, prepared_experiment.runtime_metadata, training_data.diagnostics, training_data.split_summary, outcome, metrics, predictions, training_data.input_size, components.positive_class_weight)


def print_experiment_result(result: ExperimentResult) -> None:
    print("Limiar positivo: " + format(result.test_metrics.positive_threshold, ".2f"))
    print("Métricas de teste: " + json.dumps(result.test_metrics.to_dictionary(), ensure_ascii=False))
    print("Inferência recarregada (3 amostras): " + str(result.inference_predictions))


def run_experiment(config: ExperimentConfig) -> ExperimentResult:
    prepared_experiment = prepare_experiment(config)
    training_data = prepare_training_data(config, prepared_experiment.device)
    print_data_summary(training_data)
    components = prepare_training_components(training_data, config, prepared_experiment.device)
    outcome = run_training(components, training_data, prepared_experiment)
    metrics = run_test_evaluation(components, training_data, outcome, config, prepared_experiment.device)
    persist_primary_artifacts(training_data, outcome, metrics, config)
    predictions = verify_saved_inference(training_data, config, prepared_experiment.device)
    result = build_experiment_result(prepared_experiment, training_data, components, outcome, metrics, predictions)
    save_metadata(result.to_dictionary(), config.artifacts_directory)
    print_experiment_result(result)
    return result


def results_are_identical(first_result: ExperimentResult, second_result: ExperimentResult) -> bool:
    first_values = first_result.to_dictionary()
    second_values = second_result.to_dictionary()
    keys = ["splits", "history", "test_metrics", "inference_smoke_predictions"]
    for key in keys:
        if first_values[key] != second_values[key]:
            return False
    return True


def verify_reproducibility(project_root: Path, maximum_rows: int, epochs: int) -> dict[str, object]:
    directory = project_root / "artifacts" / "f2_reproducibility"
    first_config = create_default_config(project_root)
    first_config.requested_device = "cpu"
    first_config.maximum_rows = maximum_rows
    first_config.epochs = epochs
    first_config.artifacts_directory = directory / "run_1"
    second_config = create_default_config(project_root)
    second_config.requested_device = "cpu"
    second_config.maximum_rows = maximum_rows
    second_config.epochs = epochs
    second_config.artifacts_directory = directory / "run_2"
    first_result = run_experiment(first_config)
    second_result = run_experiment(second_config)
    if not results_are_identical(first_result, second_result):
        raise AssertionError("Execuções CPU com a mesma semente divergiram.")
    summary: dict[str, object] = {}
    summary["device"] = "cpu"
    summary["maximum_rows"] = maximum_rows
    summary["epochs"] = epochs
    summary["result"] = "identical deterministic CPU runs"
    save_json_file(directory / "reproducibility_check.json", summary)
    return summary


def verify_cuda(project_root: Path, maximum_rows: int, epochs: int) -> ExperimentResult:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA não está disponível neste ambiente.")
    config = create_default_config(project_root)
    config.requested_device = "cuda"
    config.maximum_rows = maximum_rows
    config.epochs = epochs
    config.artifacts_directory = project_root / "artifacts" / "f2_cuda_validation"
    return run_experiment(config)


## 11. Verificações automáticas

Casos sintéticos confirmam F2, escolha de limiar, aplicação do limiar e o mínimo de 30 épocas.

In [ ]:
def run_synthetic_checks() -> None:
    model = MLP(4, [32, 16], 1, 0.0)
    features = torch.zeros((3, 4), dtype=torch.float32)
    validate_model_output(model, features, 1, torch.device("cpu"))

    logits = torch.tensor([[0.0], [1.0]], dtype=torch.float32)
    probabilities = convert_logits_to_positive_probabilities(logits)
    expected_probabilities = torch.sigmoid(logits).reshape(-1)
    if not torch.allclose(probabilities, expected_probabilities):
        raise AssertionError("Sigmoid não foi aplicado ao logit único.")
    threshold_predictions = create_predictions_from_threshold(
        probabilities.numpy(),
        0.50,
    )
    if threshold_predictions.tolist() != [1, 1]:
        raise AssertionError("O limiar não foi aplicado às probabilidades.")

    training_target = pd.Series([0, 0, 0, 1])
    positive_weight = calculate_positive_class_weight(training_target)
    if abs(float(positive_weight.item()) - 3.0) > 1e-12:
        raise AssertionError("pos_weight não foi calculado somente a partir do treino.")

    actual_labels = np.asarray([1, 1, 0], dtype=np.int64)
    predicted_labels = np.asarray([1, 1, 1], dtype=np.int64)
    metrics = calculate_positive_class_metrics(actual_labels, predicted_labels, 2.0)
    expected_f2 = 10.0 / 11.0
    if abs(metrics.f2 - expected_f2) > 1e-12:
        raise AssertionError("O cálculo de F2 falhou.")

    config = create_default_config(Path.cwd())
    probabilities = np.asarray([0.40, 0.80, 0.30, 0.10], dtype=np.float32)
    labels = np.asarray([1, 1, 0, 0], dtype=np.int64)
    selection = select_best_threshold(EpochResult(0.0, labels, probabilities), config)
    if abs(selection.threshold - 0.40) > 1e-12:
        raise AssertionError("A seleção de limiar não escolheu o melhor F2.")

    if should_stop_early(29, 8, 30, 8):
        raise AssertionError("Early stopping ocorreu antes da época mínima.")
    if not should_stop_early(30, 8, 30, 8):
        raise AssertionError("Early stopping não ocorreu após a época mínima.")
    print("Verificações sintéticas aprovadas.")


run_synthetic_checks()


## 12. Estratégia de melhoria: F2, limiar e saída binária única

A MLP usa `Entrada → 32 → 16 → 1`. A única saída é um logit: `sigmoid` a converte em probabilidade positiva apenas para validação, teste e inferência. `BCEWithLogitsLoss` recebe `pos_weight`, calculado somente com o treino, porque diabetes é minoritária. O maior F2 de validação escolhe checkpoint e limiar; o teste permanece isolado.

## 13. Configurar a execução

O treino final usa o dataset completo, a arquitetura `Entrada → 32 → 16 → 1`, no mínimo 30 e no máximo 200 épocas.

In [ ]:
PROJECT_ROOT = Path.cwd()
config = create_default_config(PROJECT_ROOT)
config.maximum_rows = None
config.epochs = 200
config.minimum_epochs = 30
config.hidden_dimensions = [32, 16]
config.output_size = 1
config.use_class_weights = True
config.artifacts_directory = PROJECT_ROOT / "artifacts" / "f2_full_dataset"
validate_config(config)
config_to_dictionary(config)


## 14. Executar o experimento

A chamada prepara os dados, calibra o limiar na validação, treina, avalia uma única vez no teste e salva os artefatos.

In [ ]:
result = run_experiment(config)
result.to_dictionary()


## 15. Validações opcionais

As chamadas permanecem comentadas porque treinam novamente.

In [ ]:
# reproducibility_result = verify_reproducibility(PROJECT_ROOT, maximum_rows=3000, epochs=30)
# cuda_result = verify_cuda(PROJECT_ROOT, maximum_rows=3000, epochs=30)
